# EnerGIS Framework - Runner

Haupteinstiegspunkt für Optimierungsläufe mit dem EnerGIS Planning Framework.

## Übersicht

Dieses Notebook führt einen vollständigen Optimierungslauf durch:
- **Perfect Forecast (PF)**: Optimale Dimensionierung über den gesamten Zeitraum
- **Rolling Horizon (RH)**: Operative Planung mit rollendem Horizont
- **Model Predictive Control (MPC)**: RH mit Forecast-Updates
- **PF → RH/MPC**: Kombinierter Workflow mit Design-Fixierung

## Quick Start

1. Alle Zellen mit **Run All** ausführen
2. Bei Bedarf Config-Pfade in Zelle 3 anpassen
3. Ergebnisse werden automatisch in `saved_workflows/` gespeichert
4. Dashboard wird optional am Ende angezeigt

---

## 1. Setup & Imports

In [ ]:
# Minimal-Bootstrap: Füge Projekt-Root zu sys.path hinzu
import sys
from pathlib import Path

# Finde Projekt-Root
current = Path.cwd()
for candidate in [current] + list(current.parents):
    if (candidate / 'energis').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

# Auto-Setup mit notebook_helpers
from energis.io.notebook_helpers import setup_notebook_environment

PROJECT_ROOT = setup_notebook_environment()
print("\n✅ Setup abgeschlossen")

In [ ]:
# Imports
from datetime import datetime
from energis.run import rolling_horizon as rh
from energis.io.notebook_helpers import (
    save_workflow_run,
    display_workflow_summary,
    display_kpi_summary
)

print("✅ Imports erfolgreich")

## 2. Konfiguration

Die Konfiguration erfolgt über YAML-Dateien, die in der angegebenen Reihenfolge gemerged werden.
Spätere Dateien überschreiben frühere Einträge.

### Vordefinierte Konfigurationen:

| Option | Beschreibung | Laufzeit |
|--------|--------------|----------|
| `CONFIG_STANDARD` | PF+RH Workflow mit Baseline-System | ~30-60 min |
| `CONFIG_STADTBACH_RH` | Stadtbach-System mit Rolling Horizon | ~20-40 min |
| `CONFIG_TEST` | **Schneller Test (1 Tag, PF)** | ~1-5 min |

### Zum Wechseln:
Ändere in der nächsten Zelle `CONFIG_PATHS = CONFIG_TEST` zu einer anderen Option.

### Dateien pro Konfiguration:
- `base.yaml` - Basis-Einstellungen (Solver, Zeitschritt)
- `tech_catalog.yaml` - Technologie-Katalog (Komponenten-Definitionen)
- `default.site.yaml` - Standort-Daten (Input-Daten, Zeitzone)
- `*.system.yaml` - System-Topologie (Komponenten, Kapazitäten)
- `*.scenario.yaml` - Szenario (Run-Mode, RH-Parameter, Horizont)

In [ ]:
# =============================================================================
# KONFIGURATION AUSWÄHLEN
# =============================================================================

# Option 1: Standard PF+RH Workflow (vollständiges Jahr)
CONFIG_STANDARD = [
    'configs/base.yaml',
    'configs/tech_catalog.yaml',
    'configs/sites/default.site.yaml',
    'configs/systems/baseline.system.yaml',
    'configs/scenarios/pf_then_rh.workflow.scenario.yaml',
]

# Option 2: Stadtbach System mit Rolling Horizon
CONFIG_STADTBACH_RH = [
    'configs/base.yaml',
    'configs/tech_catalog.yaml',
    'configs/sites/default.site.yaml',
    'configs/systems/stadtbach.system.yaml',
    'configs/scenarios/rolling_horizon_only.scenario.yaml',
]

# Option 3: Schneller Test (1 Tag, Perfect Foresight)
CONFIG_TEST = [
    'configs/base.yaml',
    'configs/tech_catalog.yaml',
    'configs/sites/default.site.yaml',
    'configs/systems/stadtbach.system.yaml',
    'configs/scenarios/test_1week.scenario.yaml',
]

# =============================================================================
# AKTIVE KONFIGURATION WÄHLEN (hier ändern!)
# =============================================================================
CONFIG_PATHS = CONFIG_TEST  # <- Ändern zu CONFIG_STANDARD oder CONFIG_STADTBACH_RH

# Optional: Overrides für spezifische Parameter
# Beispiele:
# - Run-Mode ändern: {'scenario': {'run_mode': 'PF_ONLY'}}
# - Solver ändern: {'run': {'solver': 'glpk'}}
# - RH-Parameter: {'scenario': {'rolling_horizon': {'heat_horizon_hours': 72}}}
OVERRIDES = None

# =============================================================================
# Config-Dateien prüfen
# =============================================================================
print("📋 Aktive Konfiguration:")
all_exist = True
for cfg_path in CONFIG_PATHS:
    full_path = PROJECT_ROOT / cfg_path
    exists = full_path.exists()
    symbol = '✅' if exists else '❌'
    print(f"  {symbol} {cfg_path}")
    if not exists:
        all_exist = False

if not all_exist:
    raise FileNotFoundError("Nicht alle Config-Dateien gefunden!")

print("\n✅ Konfiguration OK")

## 3. Workflow ausführen

Der Workflow führt die Optimierung gemäß der konfigurierten Run-Mode aus:

- **PF_ONLY**: Nur Perfect Forecast
- **RH_ONLY**: Nur Rolling Horizon
- **MPC_ONLY**: Nur Model Predictive Control (mit Forecasts)
- **PF_THEN_RH**: PF für Dimensionierung, dann RH mit fixiertem Design
- **PF_THEN_MPC**: PF für Dimensionierung, dann MPC mit fixiertem Design

In [ ]:
%%time
print("="*70)
print("🚀 STARTE OPTIMIERUNG")
print("="*70)
print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

try:
    workflow = rh.run_workflow(CONFIG_PATHS, overrides=OVERRIDES)
    
    print("\n" + "="*70)
    print("✅ OPTIMIERUNG ERFOLGREICH")
    print("="*70)
    print(f"\n📊 Workflow: {' → '.join(workflow.plan.steps)}")
    
    optimization_success = True
    
except Exception as e:
    print("\n" + "="*70)
    print("❌ FEHLER")
    print("="*70)
    print(f"\nFehler: {e}\n")
    
    import traceback
    traceback.print_exc()
    
    workflow = None
    optimization_success = False

## 4. Workflow speichern & exportieren

Speichert den Workflow mit allen Ergebnissen, Metadaten und Plots in `saved_workflows/`.

In [ ]:
if optimization_success and workflow:
    # Workflow-Name und Beschreibung (anpassbar)
    WORKFLOW_NAME = "Baseline Simulation"
    WORKFLOW_DESCRIPTION = "PF + RH Optimierung mit Standard-Konfiguration"
    
    # Workflow speichern (inkl. CSV, PDF, SVG Exports)
    workflow_dir = save_workflow_run(
        workflow,
        name=WORKFLOW_NAME,
        description=WORKFLOW_DESCRIPTION,
        config_paths=CONFIG_PATHS
    )
    
    print(f"\n💡 Dashboard anzeigen:")
    print(f"   • In diesem Notebook: Siehe Zelle 7")
    print(f"   • In interactive_dashboard.ipynb: Workflow auswählen")
    
else:
    print("⚠️  Workflow-Speicherung übersprungen (Optimierung fehlgeschlagen)")

## 5. Zusammenfassung

Zeigt die wichtigsten Kennzahlen aus dem Optimierungslauf.

In [ ]:
if optimization_success and workflow:
    # Zusammenfassung anzeigen
    display_workflow_summary(workflow)
else:
    print("⚠️  Keine Ergebnisse verfügbar")

## 6. Key Performance Indicators

Detaillierte KPI-Analyse mit Kostenaufschlüsselung und Komponentenauslastung.

In [ ]:
if optimization_success and workflow:
    # Detaillierte KPI-Analyse
    display_kpi_summary(workflow)
else:
    print("⚠️  Keine KPIs verfügbar")

## 7. Dashboard zur Visualisierung

Um die gespeicherten Simulationsergebnisse zu visualisieren, verwende das **Standalone Dashboard**.

### 🎛️ Dashboard starten:

**Option 1: Python-Skript (empfohlen)**
```bash
python start_dashboard.py
```

**Option 2: Workflow Browser Notebook**
```bash
panel serve notebooks/workflow_browser.ipynb --show
```

Das Dashboard läuft unabhängig von der Simulation und lädt automatisch alle gespeicherten Workflows aus `saved_workflows/`.

In [ ]:
print("📊 Simulationsergebnisse wurden gespeichert!")
print("\n🎛️ Zum Visualisieren starte das Dashboard:")
print("   python start_dashboard.py")
print("\nOder verwende das Workflow Browser Notebook:")
print("   panel serve notebooks/workflow_browser.ipynb --show")
print("\n💡 Das Dashboard lädt automatisch alle Workflows aus 'saved_workflows/'")

---

## 📚 Weitere Informationen

- **Dokumentation**: `README.md`, `ARCHITECTURE_V2.md`
- **Methodologie**: `docs/methodology.md`
- **CLI-Nutzung**: `python -m energis.run.rolling_horizon --help`
- **Andere Notebooks**:
  - `interactive_dashboard.ipynb` - Dashboard mit gespeicherten Workflows
  - `scenario_studio.ipynb` - Interaktive Szenario-Analyse

## 🌐 Dashboard als Webapp

Um das Dashboard als eigenständige Webapp zu starten:

```bash
panel serve runner.ipynb --show
# Oder auf spezifischem Port:
panel serve runner.ipynb --port 5006 --show
```

---